# Label extraction

- **Part A: calibration review** how well does the labeler agree with the 58 gold rows?
- **Part B: pseudo-labeled dataset QA** QA of the fully-labeled dataset the Luigi pipeline produces (`train_labeled_v2.csv` + per-UID evidence files). Set `RUN_ID` to your full run when it's done.

## 1. Notebook setup

### 1.1. Imports

In [6]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import configuration as config
from data_pipeline.label_extraction import config  # noqa: E402

### 1.2. Configuration

In [12]:
CALIBRATION_RUN = '20260905_001'

## 2. Part A: calibration review

In [15]:
# Load cached calibration rows and gold labels
cal_dir = Path('../data/pipeline/label_extraction') / CALIBRATION_RUN / 'calibration_rows'
raw = pd.read_csv(config.TRAIN_CSV)

label_cols = [c for c in raw.columns if c not in ('StudyInstanceUID', 'Report')]
gold_mask = raw[label_cols].notna().any(axis=1)
gold = raw[gold_mask].set_index('StudyInstanceUID')
files = sorted(cal_dir.glob('*.json'))

print(f'{len(files)} cached calibration rows vs {len(gold)} gold rows')

58 cached calibration rows vs 58 gold rows


In [16]:
# Load cached per-UID calibration predictions: each file is a dict of
# condition -> [value, evidence], keyed by the UID (filename stem).
rows = []

for f in files:
    uid = f.stem
    pred = json.loads(f.read_text())
    rec = {'StudyInstanceUID': uid}

    for c in label_cols:
        v, _ev = pred.get(c, [0, ''])
        rec[c] = v

    rows.append(rec)

preds = pd.DataFrame(rows).set_index('StudyInstanceUID').loc[gold.index]
cmp_df = gold.join(preds, lsuffix='_label', rsuffix='_pred')
cmp_df.head()

,Report,ACL_label,MCL_label,Medial Meniscus_label,Lateral Meniscus_label,Medial OA_label,Lateral OA_label,PF OA_label,Effusion_label,Synovitis_label,...,Medial Meniscus_pred,Lateral Meniscus_pred,Medial OA_pred,Lateral OA_pred,PF OA_pred,Effusion_pred,Synovitis_pred,Baker's_pred,Contusion_pred,Fracture_pred
StudyInstanceUID,,,,,,,,,,,,,,,,,,,,,
1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149,Antecedentes Clínicos:\nEsguince rodilla. [DAT...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0,1,0,0,1,1,0,0,0,0
1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269,The study reveals normal knee joint alignment...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0,1,0,0,1,1,1,0,0,1
1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,...,1,0,1,1,1,1,0,0,0,0
1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580,"MRI of left Knee with \n-3-Plane Loc R'T, Sag ...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0,0,0,0,0,1,1,0,0,0
1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238,"In the medial compartment, there is longitudi...",1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,0


In [17]:
def kappa(g, p):
    '''Cohen's kappa.'''

    g, p = np.asarray(g), np.asarray(p)
    po = (g == p).mean()
    pg = g.mean(); pp = p.mean()
    pe = pg * pp + (1 - pg) * (1 - pp)

    return (po - pe) / (1 - pe) if pe < 1 else 1.0

In [18]:
recs = []

for c in label_cols:
    g = (cmp_df[f'{c}_label'].fillna(0) == 1).astype(int).values
    p = (cmp_df[f'{c}_pred'].fillna(0) == 1).astype(int).values

    recs.append({
        'condition': c,
        'gold_pos': int(g.sum()),
        'pred_pos': int(p.sum()),
        'accuracy': float((g == p).mean()),
        'kappa': kappa(g, p),
        'missed (0/1)': int(((g == 1) & (p == 0)).sum()),
        'false (1/0)': int(((g == 0) & (p == 1)).sum()),
    })

cmp_tbl = pd.DataFrame(recs).set_index('condition')
cmp_tbl['accuracy'] = cmp_tbl['accuracy'].map('{:.3f}'.format)
cmp_tbl['kappa'] = cmp_tbl['kappa'].map('{:.3f}'.format)

print(
    'Mean accuracy:', round(cmp_tbl['accuracy'].astype(float).mean(), 3),
    '| Mean kappa:', round(cmp_tbl['kappa'].astype(float).mean(), 3)
)

cmp_tbl

Mean accuracy: 0.816 | Mean kappa: 0.587


,gold_pos,pred_pos,accuracy,kappa,missed (0/1),false (1/0)
condition,,,,,,
ACL,24,33,0.845,0.697,0,9
MCL,9,18,0.845,0.580,0,9
Medial Meniscus,26,28,0.862,0.723,3,5
Lateral Meniscus,23,23,0.828,0.640,5,5
Medial OA,15,19,0.897,0.752,1,5
Lateral OA,11,16,0.845,0.570,2,7
PF OA,21,25,0.793,0.570,4,8
Effusion,35,51,0.724,0.346,0,16
Synovitis,27,17,0.690,0.361,14,4


In [20]:
# Mismatches per condition (condition = the condition you want to audit)
cond = 'Effusion'  # change me: any of label_cols

for cond in ['Effusion', 'Synovitis', 'Contusion']:
    g = (cmp_df[f'{cond}_label'].fillna(0) == 1).astype(int).values
    p = (cmp_df[f'{cond}_pred'].fillna(0) == 1).astype(int).values

    mm = cmp_df[(g == 1) & (p == 0) | (g == 0) & (p == 1)].copy()

    mm['gold'] = mm[f'{cond}_label'].fillna(0).astype(int)
    mm['pred'] = mm[f'{cond}_pred'].fillna(0).astype(int)
    mm['type'] = np.where((mm['gold'] == 1) & (mm['pred'] == 0), 'missed', 'false')

    print(f'{cond}: {len(mm)} mismatches -> {dict(mm["type"].value_counts())}')
    display(mm[['gold', 'pred', 'type']])

Effusion: 16 mismatches -> {'false': np.int64(16)}


,gold,pred,type
StudyInstanceUID,,,
1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269,0,1,false
1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052,0,1,false
1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756,0,1,false
1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803,0,1,false
1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891,0,1,false
1.2.826.0.1.3680043.8.498.15593638897292057356864060466120253309,0,1,false
1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404,0,1,false
1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465,0,1,false
1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608,0,1,false


Synovitis: 18 mismatches -> {'missed': np.int64(14), 'false': np.int64(4)}


,gold,pred,type
StudyInstanceUID,,,
1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052,1,0,missed
1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580,0,1,false
1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935,1,0,missed
1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756,0,1,false
1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945,1,0,missed
1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492,1,0,missed
1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192,1,0,missed
1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911,1,0,missed
1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945,1,0,missed


Contusion: 14 mismatches -> {'false': np.int64(11), 'missed': np.int64(3)}


,gold,pred,type
StudyInstanceUID,,,
1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935,1,0,missed
1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840,0,1,false
1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819,0,1,false
1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492,1,0,missed
1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803,0,1,false
1.2.826.0.1.3680043.8.498.13267780356245120052411517053322874891,0,1,false
1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192,0,1,false
1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711,0,1,false
1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187,0,1,false


## Part B: pseudo-labeled dataset QA

Set `RUN_ID` below to your full run (the one from `pipeline.py build --run ...`). Everything here reads local files only, no API calls.

In [ ]:
# Set this to your full run id, e.g. '20260705_001'
RUN_ID = None

merged_f = config.OUTPUT_ROOT / RUN_ID / config.MERGED_CSV_NAME \
    if RUN_ID else None

print(
    'merged dataset:', merged_f, 'exists:',
    merged_f.exists() if merged_f is not None else 'set RUN_ID first'
)

merged = pd.read_csv(merged_f) if merged_f and merged_f.exists() else None

if merged is not None:
    print('label_source:\n', merged['label_source'].value_counts().to_string())
    pseudo = merged[merged['label_source'] == config.LABEL_SOURCE_PSEUDO]
    print('\npseudo_status:\n', pseudo['pseudo_status'].value_counts().to_string())
    failed_f = config.OUTPUT_ROOT / RUN_ID / 'failed_uids.txt'

    if failed_f.exists():
        print('\nfailed_uids.txt lines:',
              sum(1 for _ in open(failed_f, encoding='utf-8')))

In [ ]:
# Pseudo-label distribution: how 'abnormal' does the pseudo-labeled cohort
# look? (Sanity check: expect a skewed, mostly-zero distribution.)
if merged is not None:
    pseudo = merged[merged['label_source'] == config.LABEL_SOURCE_PSEUDO].copy()
    ok = ~pseudo['pseudo_status'].isin(['failed', 'unparseable'])
    pos = (pseudo.loc[ok, label_cols].fillna(0) == 1).sum()

    dist = pd.DataFrame({
        'condition': label_cols,
        'n_pseudo': int(ok.sum()),
        'positives': pos.values,
        'positive_rate': (pos / ok.sum()).map(lambda x: f'{x:.1%}'),
    }).set_index('condition')

    print(
        'mean positives per report:',
        round(pseudo.loc[ok, label_cols].fillna(0).sum(axis=1).mean(), 2)
    )

    dist

### Spot-check: 20 random pseudo-labeled reports with evidence quotes

The per-UID JSON files under `labels/` carry the evidence strings the model quoted; these are the raw material for manual QA. Sample 20 and eyeball whether the quotes actually support the labels.

In [ ]:
if merged is None:
    raise SystemExit('Set RUN_ID first (cell above).')

pseudo = merged[merged['label_source'] == config.LABEL_SOURCE_PSEUDO]
pseudo = pseudo[pseudo['pseudo_status'] == 'pseudo']
raw = pd.read_csv(config.TRAIN_CSV).set_index('StudyInstanceUID')
labs_dir = config.OUTPUT_ROOT / RUN_ID / 'labels'

rng = np.random.default_rng(42)
pick = rng.choice(pseudo['StudyInstanceUID'].values, size=min(20, len(pseudo)))

records = []

for uid in pick:
    rec = json.loads((labs_dir / f'{uid}.csv').read_text())
    pos = {c: rec[f'{c}_evidence'] for c in label_cols if rec.get(c) == 1}

    records.append({
        'uid': uid[:22] + '...',
        'n_positives': len(pos),
        'positives': ', '.join(pos) or '-',
        'evidence': ' | '.join(f'[{c}] {e}' for c, e in pos.items()) or '-',
        'report': str(raw.loc[uid, 'Report'])[:500],
    })

sp = pd.DataFrame(records)

with pd.option_context('display.max_colwidth', 120):
    sp